# Data Reference Similarity

In [ ]:
import evaluate
from bert_score import score
from tqdm import tqdm
import pandas as pd

seed=42
pred_feedback = pd.read_pickle(f"./new_results/Llama-3.1-8B-Instruct/dpo/criteria/seed_42/feedback_predictions.pkl")
human_feedback = pd.read_pickle(f"./new_results/Llama-3.1-8B-Instruct/dpo/criteria/seed_42/feedback_true_labels.pkl")

rouge = evaluate.load('rouge')
meteor = evaluate.load('meteor')
bleu = evaluate.load('bleu')



# ROUGE 계산 함수
def calculate_score(reference_texts, generated_texts):
    
    
    rouge_scores = []
    meteor_scores = []
    bleu_scores = []
    bert_scores = []
    
    for reference, generated in tqdm(zip(reference_texts, generated_texts), total=len(reference_texts)):
        rouge_score = rouge.compute(predictions=[generated],
                              references=[reference],
                                use_stemmer=True)
        meteor_score = meteor.compute(predictions=[generated],
                              references=[reference])
        bleu_score = bleu.compute(predictions=[generated],
                                references=[reference])
        

        rouge_scores.append(rouge_score["rougeL"].item())
        meteor_scores.append(meteor_score["meteor"].item())
        bleu_scores.append(bleu_score["bleu"])
        

    
    return rouge_scores, meteor_scores, bleu_scores

rouge_scores, meteor_scores, bleu_scores = calculate_score(human_feedback, pred_feedback)

dpo_bert_scores = score(pred_feedback, human_feedback, lang="en", rescale_with_baseline=True)
